# Complete time series and recursive CTEs

Practice notebook for the datafromzero SQL lesson **Complete time series and recursive CTEs**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Predict, then see the gap

Before you run it: the period has 181 days. How many rows will this query return? Write your guess in a comment first.

In [ ]:
con.sql("""
SELECT CAST(ordered_at AS DATE) AS order_day, COUNT(*) AS cash_orders
FROM orders
WHERE payment_method = 'cash' AND status = 'completed'
GROUP BY CAST(ordered_at AS DATE)
ORDER BY order_day
""").show()

### Step 2. Generate every day with a recursive CTE

In [ ]:
con.sql("""
WITH RECURSIVE spine(order_day) AS (
  SELECT DATE '2026-01-01'
  UNION ALL
  SELECT order_day + 1 FROM spine WHERE order_day < DATE '2026-06-30'
)
SELECT COUNT(*) AS days, MIN(order_day) AS first_day, MAX(order_day) AS last_day
FROM spine
""").show()

### Step 3. Join the facts onto the spine

In [ ]:
con.sql("""
WITH RECURSIVE spine(order_day) AS (
  SELECT DATE '2026-01-01'
  UNION ALL
  SELECT order_day + 1 FROM spine WHERE order_day < DATE '2026-06-30'
),
cash_daily AS (
  SELECT CAST(ordered_at AS DATE) AS order_day, COUNT(*) AS cash_orders
  FROM orders
  WHERE payment_method = 'cash' AND status = 'completed'
  GROUP BY CAST(ordered_at AS DATE)
)
SELECT s.order_day, c.cash_orders
FROM spine AS s
LEFT JOIN cash_daily AS c ON c.order_day = s.order_day
ORDER BY s.order_day
LIMIT 6
""").show()

### Step 4. Count, and turn NULL into 0

In [ ]:
con.sql("""
WITH RECURSIVE spine(order_day) AS (
  SELECT DATE '2026-01-01'
  UNION ALL
  SELECT order_day + 1 FROM spine WHERE order_day < DATE '2026-06-30'
),
cash_daily AS (
  SELECT CAST(ordered_at AS DATE) AS order_day, COUNT(*) AS cash_orders
  FROM orders
  WHERE payment_method = 'cash' AND status = 'completed'
  GROUP BY CAST(ordered_at AS DATE)
)
SELECT COUNT(*) AS days,
       COUNT(*) - COUNT(c.cash_orders) AS empty_days,
       SUM(COALESCE(c.cash_orders, 0)) AS cash_orders,
       ROUND(AVG(COALESCE(c.cash_orders, 0)), 2) AS avg_cash_day
FROM spine AS s
LEFT JOIN cash_daily AS c ON c.order_day = s.order_day
""").show()

### Step 5. Verify without the spine

In [ ]:
con.sql("""
SELECT COUNT(*) AS cash_orders FROM orders WHERE payment_method = 'cash' AND status = 'completed'
""").show()

### Step 6. A shorter way to the same spine

In [ ]:
con.sql("""
SELECT COUNT(*) AS days,
       MIN(CAST(generate_series AS DATE)) AS first_day,
       MAX(CAST(generate_series AS DATE)) AS last_day
FROM generate_series(TIMESTAMP '2026-01-01', TIMESTAMP '2026-06-30', INTERVAL 1 DAY)
""").show()

### Step 7. Read a recursive CTE on a hierarchy: who did customer 24 bring in?

In [ ]:
con.sql("""
WITH RECURSIVE chain(customer_id, referred_by, depth) AS (
  SELECT customer_id, referred_by, 0
  FROM customers
  WHERE customer_id = 24
  UNION ALL
  SELECT c.customer_id, c.referred_by, chain.depth + 1
  FROM customers AS c
  JOIN chain ON c.referred_by = chain.customer_id
)
SELECT * FROM chain
ORDER BY depth, customer_id
""").show()

In [ ]:
con.sql("""
SELECT customer_id, referred_by FROM customers WHERE referred_by IN (24, 30, 34, 80)
""").show()

## Exercise 1. How many days had no completed mobile order?

The spine and the daily counts are done for you. Replace the last line of the query with the final `SELECT`: the same four columns as step 4 (days, empty days, mobile orders, average per day), for mobile.

In [ ]:
con.sql("""
WITH RECURSIVE spine(order_day) AS (
  SELECT DATE '2026-01-01'
  UNION ALL
  SELECT order_day + 1 FROM spine WHERE order_day < DATE '2026-06-30'
),
mobile_daily AS (
  SELECT CAST(ordered_at AS DATE) AS order_day, COUNT(*) AS mobile_orders
  FROM orders
  WHERE payment_method = 'mobile' AND status = 'completed'
  GROUP BY CAST(ordered_at AS DATE)
)
SELECT * FROM mobile_daily LIMIT 3  -- replace this line with your final SELECT
""").show()

## Exercise 2. Harbor's orders per day around its opening

Before you run anything, write one sentence in the cell below: how many of the 14 days (March 25 to April 7) will show 0, and what will each 0 mean? Then build a spine for those 14 days and join Harbor's order counts (`store_id` 3, any status) onto it.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague joins the spine straight to `orders` and counts with `COUNT(*)`. Run it and compare the second number with step 5. What went wrong, and what is the fix?

In [ ]:
con.sql("""
WITH RECURSIVE spine(order_day) AS (
  SELECT DATE '2026-01-01'
  UNION ALL
  SELECT order_day + 1 FROM spine WHERE order_day < DATE '2026-06-30'
),
daily AS (
  SELECT s.order_day, COUNT(*) AS cash_orders
  FROM spine AS s
  LEFT JOIN orders AS o
    ON CAST(o.ordered_at AS DATE) = s.order_day
   AND o.payment_method = 'cash' AND o.status = 'completed'
  GROUP BY s.order_day
)
SELECT COUNT(*) AS days, SUM(cash_orders) AS cash_orders FROM daily
""").show()

In [ ]:
# Your corrected query: